# Audio fingerprinting workbook

## Generating Peak Pair Hashes for query and reference files

### Constellations

Transform audio file into freq domain

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
from main import *

Spectrogram parameters

In [ ]:
sample_rate = SR
window_length = WINDOW_LENGTH
hop_length = HOP_LENGTH
n_fft = N_FFT

In [ ]:
reference_song_id = '086795'
query_song_id = reference_song_id

reference_spectrogram = to_spectrogram('../audio_data/database/' + reference_song_id + '.mp3', sample_rate, window_length, hop_length, n_fft)
plt.pcolormesh(reference_spectrogram)

### find features – prominent peaks

Peak filter parameters

In [ ]:
peak_level_threshold_db = 2
neighbourhood_size = 20

In [ ]:
# how we define prominent peaks (in constellation())
# local_max = maximum_filter(spectrogram, mode="constant", size=neighbourhood_size)
# is_peak = (spectrogram == local_max) & (spectrogram > level_threshold)

In [ ]:
reference_peaks = constellation(reference_spectrogram, peak_level_threshold_db, neighbourhood_size)
plt.pcolormesh(reference_spectrogram)
plt.scatter(reference_peaks[:,1], reference_peaks[:,0], marker='+', color='r')


### Generate peak pairing hashes

pairing hash filter constants

In [ ]:
fan_out = 10
min_time_delta = 1
max_time_delta = 200

In [ ]:
reference_hashes = generate_hashes(reference_peaks, fan_out, min_time_delta, max_time_delta)

# hash: [hash_val, t_anchor]
# hash_val = (int(f_anchor & 0x3FFF) << 18) | (int(f_target & 0x3FF) << 8) | int(delta_t & 0xFF)

hash_index = 1

reference_hash = reference_hashes[hash_index]
t_anchor = reference_hash[1]

# decode information stored in hash
f_anchor = reference_hash[0] >> 18
f_target = reference_hash[0] >> 8 & 0x3FF
delta_t = reference_hash[0] & 0xFF
t_target = t_anchor + delta_t


plt.scatter(reference_peaks[:,1], reference_peaks[:,0])
plt.scatter([t_anchor, t_target], [f_anchor, f_target], marker='x')


## Match query Files

let's have a look at the constellations

In [ ]:
spectrogram = to_spectrogram('../audio_data/query/' + reference_song_id + '_query.mp3', sample_rate, window_length, hop_length, n_fft)

query_peaks = constellation(spectrogram, peak_level_threshold_db, neighbourhood_size)

plt.pcolormesh(spectrogram)
plt.scatter(query_peaks[:,1], query_peaks[:,0], marker='+', color='r')

We can't match the reference peaks with the query peaks just yet

In [ ]:
plt.scatter(query_peaks[:,1], query_peaks[:,0], marker='+', color='r')
plt.scatter(reference_peaks[:,1], reference_peaks[:,0])

### Finding identical hash values in query and reference data

In [ ]:
query_hashes = generate_hashes(query_peaks, fan_out, min_time_delta, max_time_delta)
query_hash_values = [hash[0] for hash in query_hashes]
query_min = np.min(query_hash_values)
query_max = np.max(query_hash_values)

reference_hash_values = [hash[0] for hash in reference_hashes if hash[0] > query_min and hash[0] < query_max]

plt.scatter(reference_hash_values, np.zeros_like(reference_hash_values), marker='_')
plt.scatter(query_hash_values, np.zeros_like(query_hash_values), marker='|')


identical_hashes = np.sort([query_hash for query_hash in query_hash_values if query_hash in reference_hash_values])
plt.scatter(identical_hashes, np.zeros_like(identical_hashes), marker='*')


for now, don't consider absolute time to find matches

## Scoring with time delta

absolute time of found peaks is considered in a second step of the algorithm

In [ ]:
ref_fingerprint = {}
fingerprints(reference_hashes, reference_song_id, ref_fingerprint)
matches = np.array(find_matches(query_hashes, ref_fingerprint))
plt.scatter(matches[:,2], matches[:,1])


In [ ]:
time_diff_counts = time_diff_histogram(find_matches(query_hashes, ref_fingerprint))
x_values = [int(key[1]) for key in time_diff_counts]
counts = list(time_diff_counts.values())
max_count_time_diff = max(time_diff_counts, key=time_diff_counts.get)[1]
plt.bar(x_values, counts)
print(max_count_time_diff)

In [ ]:
query_peaks = constellation(spectrogram, peak_level_threshold_db)
plt.scatter(reference_peaks[:,1], reference_peaks[:,0])
plt.scatter(query_peaks[:,1] + max_count_time_diff, query_peaks[:,0], marker='x')